# Chapter 15 &mdash; Tile Construction: Simulating a TM Through a Peephole

**Concept 3 of the Chapter 15 decomposition:** *Tile Construction: Simulating a TM Through a Peephole*

Walk $\delta$ and $\Gamma$ to generate a finite tile set that can only be assembled one way.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-Tile-Construction/Concept-Tile-Construction.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The tiles are generated **mechanically** from $\delta$ and $\Gamma$. Four families:

* a **start tile** $\left[\frac{\#}{\# C_0 \#}\right]$, which puts the bottom one
  configuration ahead;
* **copy tiles** $\left[\frac{a}{a}\right]$ for every tape symbol, to carry unchanged
  cells forward;
* **transition tiles**, one per $\delta$ entry, e.g. $\delta(q,a)=(p,b,R)$ gives
  $\left[\frac{qa}{bp}\right]$;
* **cleanup tiles** that let the accepting configuration eat the tape so the two rows
  can finally meet.

The key insight is **locality**: a TM step changes only a **peephole** of two or three
cells. Everything else is copied. So a finite tile set suffices, however long the
computation.

## 2. Definitions

### Generate the tiles from a TM

In [ ]:
# --- a Post Correspondence solver, bounded by tile count ----------------
# An instance is a list of (top, bottom) dominoes.  A solution is a
# non-empty sequence of indices whose concatenated tops equal its bottoms.
def pcp_search(tiles, maxlen=8):
    from collections import deque
    # a partial solution is (indices, top, bottom); one side is a prefix
    # of the other, or the partial is dead
    dq = deque([((i,), t, b) for i, (t, b) in enumerate(tiles)])
    while dq:
        idx, top, bot = dq.popleft()
        if top == bot:
            return list(idx)
        if len(idx) >= maxlen:
            continue
        if not (top.startswith(bot) or bot.startswith(top)):
            continue                                  # dead: they diverge
        for j, (t, b) in enumerate(tiles):
            dq.append((idx + (j,), top + t, bot + b))
    return None

def pcp_check(tiles, sol):
    top = ''.join(tiles[i][0] for i in sol)
    bot = ''.join(tiles[i][1] for i in sol)
    return top == bot, top, bot

def show_tiles(tiles):
    print("   " + "  ".join("[%s/%s]" % t for t in tiles))


def tm_tiles(T):
    G = sorted(T["Gamma"])
    tiles, why = [], []
    def add(t, b, w):
        tiles.append((t, b)); why.append(w)
    # copy tiles
    for a in G:
        add(a, a, "copy tape symbol %r" % a)
    add('#', '#', "copy the separator")
    # transition tiles
    for (q, a), outs in sorted(T["Delta"].items()):
        for (p, b, d) in sorted(outs):
            if d == 'R':
                add(q + a, b + p, "delta(%s,%s) = (%s,%s,R)" % (q, a, p, b))
            elif d == 'L':
                for c in G:
                    add(c + q + a, p + c + b,
                        "delta(%s,%s) = (%s,%s,L), left neighbour %r" % (q, a, p, b, c))
            else:
                add(q + a, p + b, "delta(%s,%s) = (%s,%s,S)" % (q, a, p, b))
    return tiles, why

## 3. Tests

A small machine, and the tiles it generates.

In [ ]:
T = md2mc('''TM
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')
tiles, why = tm_tiles(T)
print("%d tiles from %d Delta entries and %d tape symbols"
      % (len(tiles), len(T["Delta"]), len(T["Gamma"])))
for (t, b), w in list(zip(tiles, why))[:12]:
    print("   [%-6s / %-6s]   %s" % (t, b, w))

The tile set is **finite** and computed without running the machine.

In [ ]:
assert len(tiles) < 100
print("tiles :", len(tiles))
print("the generator walked Delta and Gamma -- it never simulated anything")
print()
print("That matters: the REDUCTION must be computable even when the machine")
print("it describes never halts.")

**Locality:** a step changes a peephole; everything else is copied.

In [ ]:
copies = [(t, b) for (t, b), w in zip(tiles, why) if w.startswith('copy')]
moves  = [(t, b) for (t, b), w in zip(tiles, why) if w.startswith('delta')]
print("copy tiles       :", copies)
print("transition tiles :", moves)
print()
print("|copy| = %d, |transition| = %d." % (len(copies), len(moves)))
print("One transition tile per Delta entry (times |Gamma| for a left move),")
print("and one copy tile per symbol.  Finite, whatever the computation length.")

Left moves need the **left neighbour** in the peephole &mdash; hence $|\Gamma|$ copies.

In [ ]:
L = md2mc('''TM
I : 0 ; 1 , L -> I
I : . ; . , S -> F
''')
lt, lw = tm_tiles(L)
lefts = [(t, b) for (t, b), w in zip(lt, lw) if ', L)' in w or ',L)' in w or 'L),' in w]
for (t, b), w in zip(lt, lw):
    if 'L)' in w: print("   [%-6s / %-6s]   %s" % (t, b, w))
print("\nA right move sees two cells; a left move sees three.")

The construction is a **function on machine descriptions**, which is what a reduction is.

In [ ]:
for src in ['TM\nI : 0 ; 1 , R -> F\n',
            'TM\nI : 0 ; 0 , R -> I\nI : . ; . , S -> F\n']:
    TT = md2mc(src)
    tt, _ = tm_tiles(TT)
    print("  machine with %d Delta entries -> %d tiles" % (len(TT["Delta"]), len(tt)))
print("\nf(<M,w>) = this tile set.  Computable, total, and it never runs M.")

## 4. Exercises


1. Write the cleanup tiles that let the accepting configuration eat the tape.
2. Why does a left move need one tile per left-neighbour symbol?
3. How many tiles does a machine with 5 states and 3 tape symbols generate?

In [ ]:
# Your work for the exercises above.